In [1]:
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import sys

sys.path.append("../src/models/naive")
sys.path.append("../src")
from tfid_baseline import fit_tfidf_baseline, pred_tfidf_baseline
from metrics import compute_base_metrics  

In [ ]:
# Loads dataset
with open("../data/cuad/train_separate_questions.json") as f:
    raw = json.load(f)

# Flatten everything to one row (contract, paragraph, and category)
rows = []
for doc in raw["data"]:
    for p_idx, paragraph in enumerate(doc["paragraphs"]):
        for qa in paragraph["qas"]:
            category = qa["id"].split("__")[-1] if "__" in qa["id"] else qa["question"] 
            rows.append({
                "contract": doc["title"],
                "paragraph_idx": p_idx,
                "text": paragraph["context"],
                "category": category,
                "is_present": len(qa.get("answers", [])) > 0,
            })
flat_df = pd.DataFrame(rows)

In [ ]:
# Identifies the unique clause categories and assigns an index
categories = sorted(flat_df["category"].unique())
cat_to_idx = {c: i for i, c in enumerate(categories)}

In [ ]:
# Helper that builds 41 element 1d numpy array of 0s and marks index as 1 if is_present is true
def make_label_vector(group):
    vec = np.zeros(len(categories), dtype=np.float32)
    for cat in group.loc[group["is_present"], "category"]:
        vec[cat_to_idx[cat]] = 1.0
    return vec

# Groups flattened data back by unique chunk
chunk_df = (
    flat_df.groupby(["contract", "paragraph_idx", "text"])
    .apply(make_label_vector)
    .reset_index(name="labels")
)

In [ ]:
# Splits data into 80/20 training and validation split, fits, and predicts
train_df, val_df = train_test_split(chunk_df, test_size=0.2, random_state=42)
y_train = np.stack(train_df["labels"])
y_val = np.stack(val_df["labels"])

vectorizer, classifier = fit_tfidf_baseline(train_df["text"].tolist(), y_train)
y_pred = pred_tfidf_baseline(vectorizer, classifier, val_df["text"].tolist())

In [ ]:
# Computes metrics
report = compute_base_metrics(y_val, y_pred, categories)
report.head(20)

,category,precision,recall,f1,support
0,Document Name_0,1.00000,1.00000,1.00000,82
1,Parties_1,1.00000,1.00000,1.00000,82
2,Parties_0,1.00000,1.00000,1.00000,82
3,Parties_2,0.96341,1.00000,0.98137,79
4,Agreement Date_0,0.94872,0.96104,0.95484,77
5,Parties_3,0.92500,0.97368,0.94872,76
6,Expiration Date_0,0.90000,0.96923,0.93333,65
7,Effective Date_0,0.86154,0.86154,0.86154,65
8,Governing Law_0,0.79747,0.98438,0.88112,64
9,Anti-Assignment_0,0.76812,0.96364,0.85484,55
